In [ ]:
from pyspark.sql import functions as F
from spark_session_config import spark
from schema import samples_data_schema

samples_enriched_df = spark.readStream.format("kafka") \
.option("kafka.bootstrap.servers", "course-kafka:9092") \
.option("subscribe", "samples-enriched") \
.option("startingOffsets", "earliest") \
.load() \
.select(F.col("value").cast("string"))

#convert from json to dataframe

parsed_df = samples_enriched_df \
.withColumn('parsed_json', F.from_json(F.col("value"), samples_data_schema)) \
.select(F.col('parsed_json.*'))

filter_df = parsed_df \
.where((F.col("speed") > 120) & (F.col("expected_gear") != F.col("gear")) & (F.col("rpm") > 6000))

data_json_df = filter_df.select(
    F.to_json(F.struct("*")).alias("value")
)

data_json_df.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "course-kafka:9092") \
    .option("topic", "alert-data") \
    .option("checkpointLocation", 's3a://pyspark/checkpoints/checkpoints/alert-data') \
    .outputMode("update") \
    .start() \
    .awaitTermination()

spark.stop()
